# banxicolab — inicio rápido

Tres preparativos, en orden y una sola vez: **instalar**, **importar** y **configurar el token**.
Después, cualquier análisis.

## 1. Instala
La celda siguiente instala la librería solo si no la tienes (en Colab, por ejemplo).

In [1]:
# Instala la librería solo si hace falta (por ejemplo, en Google Colab)
try:
    import banxicolab
except ImportError:
    %pip install -q banxicolab

## 2. Importa

In [2]:
import banxicolab as sie

print(sie.__version__)

# Guarda las descargas en disco: volver a ejecutar el notebook no gasta consultas
sie.configurar(cache="disco")

0.4.1


Banxico(token=sin resolver)

## 3. Configura tu token
El token es gratuito y se genera en <https://www.banxico.org.mx/SieAPIRest/service/v1/token>.

La **primera vez** en esta computadora, quita el `#` y ejecuta la celda: te pide el token sin mostrarlo, lo comprueba con Banxico y lo guarda. Después vuelve a poner el `#`; ya no hace falta. En Colab conviene guardarlo en *Secrets* con el nombre `BANXICO_TOKEN`.

In [3]:
# sie.guardar_token()

## 4. Ya puedes trabajar
El catálogo lista las series que tienen nombre corto.

Las gráficas son interactivas: pasa el cursor para ver los valores, arrastra para acercar y usa los botones para elegir el periodo.

In [4]:
sie.catalogo()

,id,descripcion,tema,periodicidad,unidad,agregacion
alias,,,,,,
inpc,SP1,"INPC, índice general",precios,Mensual,Índice,ultimo
inpc_subyacente,SP74625,INPC subyacente,precios,Mensual,Índice,ultimo
inpc_no_subyacente,SP74630,INPC no subyacente,precios,Mensual,Índice,ultimo
inpc_mercancias,SP74626,INPC subyacente: mercancías,precios,Mensual,Índice,ultimo
inpc_servicios,SP74628,INPC subyacente: servicios,precios,Mensual,Índice,ultimo
inpc_agropecuarios,SP56337,INPC no subyacente: agropecuarios,precios,Mensual,Índice,ultimo
inpc_energeticos,SP74631,INPC no subyacente: energéticos y tarifas del ...,precios,Mensual,Índice,ultimo
inflacion_mensual,SP30577,"INPC, variación mensual",precios,Mensual,%,ultimo
inflacion_anual,SP30578,"INPC, variación anual",precios,Mensual,%,ultimo


## Descarga
Las series diarias y las mensuales se piden por separado y se unen ya en frecuencia mensual.

In [5]:
inpc = sie.serie("inpc", "2000-01-01")
diarias = sie.descargar(["fix", "tasa_objetivo", "cetes_28"], "2000-01-01")

sie.diagnostico(diarias)

,obs,inicio,fin,frecuencia,faltantes,min,max,razon_max_min
serie,,,,,,,,
fix,6754,2000-01-03,2026-10-30,Diaria,2038,8.9428,25.1185,2.808796
tasa_objetivo,6766,2008-01-21,2026-11-03,Diaria,0,3.0000,11.2500,3.750000
cetes_28,1050,2006-09-19,2026-10-27,Semanal,6042,2.4300,11.4000,4.691358


In [6]:
mensual = sie.a_mensual(
    diarias, {"fix": "ultimo", "tasa_objetivo": "promedio", "cetes_28": "promedio"}
)
panel = mensual.join(inpc)
panel.tail()

,fix,tasa_objetivo,cetes_28,inpc
fecha,,,,
2026-07-01,17.3288,6.5,6.2175,145.169
2026-08-01,17.0147,6.5,6.2125,145.462
2026-09-01,18.0692,6.5,6.2300,146.075
2026-10-01,18.7962,6.5,6.1975,NaN
2026-11-01,NaN,6.5,NaN,NaN


## Inflación

In [7]:
tabla = sie.inflacion(inpc)
tabla.tail()

,mensual,anual,acumulada,mensual_anualizada
fecha,,,,
2026-05-01,-0.208460,3.938948,1.737252,-2.473043
2026-06-01,-0.272114,3.365977,1.460410,-3.216943
2026-07-01,0.026183,3.117630,1.486976,0.314652
2026-08-01,0.201834,3.261942,1.691811,2.449073
2026-09-01,0.421416,3.454748,2.120356,5.175863


In [8]:
sie.graficar(tabla.loc["2015":, ["anual"]].rename(columns={"anual": "Inflación anual"}),
             "Inflación anual y meta de Banxico", unidad="%", meta=True)

## Poder adquisitivo y tasa real

In [9]:
hoy = inpc.index[-1]
for anio in [2000, 2010, 2020]:
    print(f"$1,000 de enero {anio} = ${sie.a_pesos_de(1000, f'{anio}-01-01', hoy, inpc):,.2f} de {hoy:%b %Y}")

$1,000 de enero 2000 = $3,251.11 de Sep 2026
$1,000 de enero 2010 = $2,013.38 de Sep 2026
$1,000 de enero 2020 = $1,372.28 de Sep 2026


In [10]:
real = sie.tasa_real(panel["cetes_28"], tabla["anual"]).dropna()
print(f"Meses con tasa real negativa: {(real < 0).mean():.1%}")
sie.graficar(real.rename("Tasa real"), "Tasa real ex-post (Cetes 28 días)", unidad="% anual", cero=True)

Meses con tasa real negativa: 22.8%


## ¿Cuál transformación induce estacionariedad?
Requiere `statsmodels`.

In [11]:
sie.prueba_adf({
    "nivel": inpc,
    "índice base 100": sie.indice_base(inpc, "2018-01-01"),
    "z-score": sie.estandarizar(inpc),
    "Δ ln (inflación mensual)": sie.variacion(inpc, log=True),
    "Δ12 ln (inflación anual)": sie.variacion(inpc, 12, log=True),
})

,estadistico,critico_5%,p_valor,rezagos,obs,estacionaria
serie,,,,,,
nivel,2.053251,-2.870970,0.998738,12,308,False
índice base 100,2.053251,-2.870970,0.998738,12,308,False
z-score,2.053251,-2.870970,0.998738,12,308,False
Δ ln (inflación mensual),-3.695696,-2.870970,0.004174,11,308,True
Δ12 ln (inflación anual),-3.107012,-2.871388,0.026039,13,295,True
